In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
pd.set_option('display.max_columns', None)

In [ ]:
# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "df_final.csv")
df_final = pd.read_csv(file_path)

In [ ]:
# Function to clean EDMOM and EDDAD values
def clean_ed(x):
    if pd.isna(x):  # Keep NaN as is
        return x
    if isinstance(x, str):
        if x.isdigit():  # Convert string numbers to int
            return int(x)
        elif x == 'E':   # Keep 'E' as is
            return 'E'
    return x  # Return original value if it's already an integer

# Apply the cleaning function to EDMOM and EDDAD for all years
df_final['EDMOM'] = df_final['EDMOM'].apply(clean_ed)
df_final['EDDAD'] = df_final['EDDAD'].apply(clean_ed)

# Verify the cleaned values
for year in sorted(df_final['REFYR'].unique()):
    edmom_values = df_final[df_final['REFYR'] == year]['EDMOM'].unique()
    eddad_values = df_final[df_final['REFYR'] == year]['EDDAD'].unique()
    
    print(f"\nREFYR {year}:")
    print(sorted([x for x in edmom_values if x != 'E']) + (['E'] if 'E' in edmom_values else []))
    print(sorted([x for x in eddad_values if x != 'E']) + (['E'] if 'E' in eddad_values else []))


In [ ]:
import os
import pandas as pd

# Define the path and read the CSV file
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")

# Read the CSV file
ur_ba_16y = pd.read_csv(ur_ba_path)

# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})

# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year


# Assume there is a 'YEAR' column in ur_ba_16y indicating the year
# If not, check if 'BAYR' corresponds to a year column
if 'year' in ur_ba_16y.columns:
    # Compute the annual average unemployment rate
    ur_avg = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
    
    # Rename YEAR to BAYR so it matches df_final
    ur_avg = ur_avg.rename(columns={'year': 'BAYR'})
else:
    print("Error: No 'year' column found in ur_ba_16y. Please check the dataset structure.")

# Merge with df_final based on BAYR
df_final= df_final.merge(ur_avg, on='BAYR', how='left')

In [ ]:
df_final.columns

In [ ]:
# share(broad major in BA != broad major in MA) among ppl pursuing MA in each cohort [vary by cohort] ~ initial UR [vary by cohort]

# Prepare Data: 
define field_switch; Subset immediate_masters & all_masters

In [ ]:
def prepare_data(df):
    """
    Prepare data for analysis of field switching
    """
    # 1. Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    
    # 2. Create master's indicators
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3))
    df['with_masters'] = (df['DGRDG'] == 2)
    
    # 3. Remove observations with missing major information
    df = df[df['NBAMEMG'].notna() & df['NDGMEMG'].notna()]
    
    # 4. Create separate datasets for analysis
    immediate = df[df['immediate_masters']==1].copy()
    all_masters = df[df['with_masters']==1].copy()
    
    # 5. Create field switch indicators
    for dataset in [immediate, all_masters]:
        dataset['field_switch'] = (dataset['NBAMEMG'] != dataset['NDGMEMG']).astype(int)
    
    return immediate, all_masters

In [ ]:
def descriptive_analysis(immediate, all_masters):
    """
    Comprehensive descriptive analysis
    """
    print("Sample Sizes:")
    print(f"Immediate master's recipients: {len(immediate)}")
    print(f"All master's recipients: {len(all_masters)}")
    
    # Basic switching rates
    print("\nSwitching Rates:")
    print(f"Immediate master's: {immediate['field_switch'].mean():.2%}")
    print(f"All master's: {all_masters['field_switch'].mean():.2%}")

    def create_transition_matrix(df, title):
        field_names = {
            1: "Computer & Math",
            2: "Bio & Life Sciences",
            3: "Physical Sciences",
            4: "Social Sciences",
            5: "Engineering",
            6: "S&E-Related",
            7: "Non-S&E"
        }

        # Set display options to prevent wrapping
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)

        trans_matrix = pd.crosstab(
            df['NBAMEMG'], 
            df['NDGMEMG'],
            normalize='index'  # row percentages
        )

        # Rename both index and columns
        trans_matrix.index = trans_matrix.index.map(field_names)
        trans_matrix.columns = trans_matrix.columns.map(field_names)

        print(f"{title} (Row percentages)")
        print(trans_matrix.round(3))
        return trans_matrix
    
    # Field patterns
    def field_patterns(df, title):
        patterns = df.groupby('NBAMEMG').agg({
            'field_switch': ['mean', 'count'],
            'NDGMEMG': lambda x: x.mode()[0] if not x.empty else None
        }).round(3)
        
        print(f"\nField Patterns - {title}")
        print("Switch Rate = Proportion changing fields")
        print("Count = Number of observations")
        print("Most Common Destination = Most common master's field if switched")
        print(patterns)
        return patterns
    
    matrix_immediate = create_transition_matrix(immediate, "Immediate Master's")
    matrix_all = create_transition_matrix(all_masters, "All Master's")
    
    patterns_immediate = field_patterns(immediate, "Immediate Master's")
    patterns_all = field_patterns(all_masters, "All Master's")

    # Unemployment rate analysis (rest of the code remains the same)
    def ur_analysis(df, title):
        print(f"\nUnemployment Rate Analysis - {title}")
        ur_quintiles = pd.qcut(df['UR'], q=5)
        switch_by_ur = df.groupby(ur_quintiles)['field_switch'].mean()
        print("Switching rates by unemployment rate quintile:")
        print(switch_by_ur)

        # Statistical test
        from scipy import stats
        high_ur = df[df['UR'] > df['UR'].median()]['field_switch']
        low_ur = df[df['UR'] <= df['UR'].median()]['field_switch']
        t_stat, p_val = stats.ttest_ind(high_ur, low_ur)
        print(f"\nt-test p-value for high vs low unemployment: {p_val:.4f}")

    ur_analysis(immediate, "Immediate Master's")
    ur_analysis(all_masters, "All Master's")
    

    return {
        'matrix_immediate': matrix_immediate,
        'matrix_all': matrix_all,
        'patterns_immediate': patterns_immediate,
        'patterns_all': patterns_all
    }

In [ ]:
immediate, all_masters = prepare_data(df_final)

results = descriptive_analysis(immediate, all_masters)

## Analyze UR ~ field switch rate by cohort

In [ ]:
def analyze_ur_cohort_switch(immediate, all_masters, title=None):
    """
    Analyze and visualize relationship between UR, field switching, and graduation cohort
    for either immediate master's or all master's recipients
    """
    # Calculate mean UR and switch rate by cohort
    stats_by_cohort = immediate.groupby('BAYR').agg({
        'UR': 'mean',
        'field_switch': 'mean'
    }).reset_index()
    
    # Create figure with two y-axes
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()
    
    # Plot unemployment rate on left y-axis
    line1 = ax1.plot(stats_by_cohort['BAYR'], stats_by_cohort['UR'], 
                     color='blue', marker='o', linestyle='-', linewidth=2,
                     label='Unemployment Rate')
    
    # Plot switch rate on right y-axis
    line2 = ax2.plot(stats_by_cohort['BAYR'], stats_by_cohort['field_switch'], 
                     color='red', marker='s', linestyle='-', linewidth=2,
                     label='Field Switch Rate')
    
    # Customize the plot
    ax1.set_xlabel('Bachelor Graduation Year', fontsize=10)
    ax1.set_ylabel('Unemployment Rate (%)', color='blue', fontsize=10)
    ax2.set_ylabel('Field Switch Rate', color='red', fontsize=10)
    
    title = title if title else 'Unemployment Rate and Field Switching Rate by Graduation Cohort'
    plt.title(title, fontsize=12, pad=15)
    
    # Add grid
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # Combine legends
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    
    # Print correlations
    ur_corr = immediate[['BAYR', 'UR']].corr().iloc[0,1]
    switch_corr = immediate[['BAYR', 'field_switch']].corr().iloc[0,1]
    ur_switch_corr = immediate[['UR', 'field_switch']].corr().iloc[0,1]
    
    print(f"\nCorrelations:")
    print(f"Graduation year - Unemployment rate: {ur_corr:.3f}")
    print(f"Graduation year - Switch rate: {switch_corr:.3f}")
    print(f"Unemployment rate - Switch rate: {ur_switch_corr:.3f}")
    
    plt.tight_layout()
    plt.show()
    
    return stats_by_cohort

# 分别对两组进行分析
# For immediate master's
trends_immediate = analyze_ur_cohort_switch(immediate, all_masters, 
                                          title="Immediate Master's: UR and Field Switching Rate")

# For all master's
trends_all = analyze_ur_cohort_switch(all_masters, all_masters, 
                                     title="All Master's: UR and Field Switching Rate")

In [ ]:
def check_field_codes(df):
    """
    Check if there are any field codes not in the valid range
    """
    valid_codes = [11, 12, 21, 22, 23, 31, 32, 33, 34, 41, 42, 43, 44, 45,
                  51, 52, 53, 54, 55, 56, 57, 61, 62, 63, 64, 71, 72, 73, 
                  74, 75, 76]
    
    # Check NBAMENG
    unique_ba = sorted(df['NBAMENG'].unique())
    invalid_ba = [code for code in unique_ba if code not in valid_codes]
    
    # Check NDGMENG
    unique_ma = sorted(df['NDGMENG'].unique())
    invalid_ma = [code for code in unique_ma if code not in valid_codes]
    
    print("Invalid Bachelor's field codes:", invalid_ba if invalid_ba else "None")
    print("Invalid Master's field codes:", invalid_ma if invalid_ma else "None")
    
    # Print value counts
    print("\nBachelor's field code distribution:")
    print(df['NBAMENG'].value_counts().sort_index())
    print("\nMaster's field code distribution:")
    print(df['NDGMENG'].value_counts().sort_index())

# 使用函数
check_field_codes(df_final)

In [ ]:
def prepare_data_minor(df):
    """
    Prepare data for analysis of field switching
    """
    # 1. Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    
    # 2. Create master's indicators
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3))
    df['with_masters'] = (df['DGRDG'] == 2)
    
    # 3. Remove observations with missing major information
    df = df[df['NBAMENG'].notna() & df['NDGMENG'].notna()]
    
    # 4. Create separate datasets for analysis
    immediate = df[df['immediate_masters']==1].copy()
    all_masters = df[df['with_masters']==1].copy()
    
    # 5. Create field switch indicators
    for dataset in [immediate, all_masters]:
        dataset['field_switch'] = (dataset['NBAMENG'] != dataset['NDGMENG']).astype(int)
    
    return immediate, all_masters


In [ ]:
def descriptive_analysis_minor(immediate, all_masters):
    """
    Comprehensive descriptive analysis
    """
    print("Sample Sizes:")
    print(f"Immediate master's recipients: {len(immediate)}")
    print(f"All master's recipients: {len(all_masters)}")
    
    # Basic switching rates
    print("\nSwitching Rates:")
    print(f"Immediate master's: {immediate['field_switch'].mean():.2%}")
    print(f"All master's: {all_masters['field_switch'].mean():.2%}")

    def create_transition_matrix(df, title):
        field_names = {
            11: "Computer & Info Sci",
            12: "Mathematics & Stats",
            21: "Agricultural Sciences",
            22: "Biological Sciences",
            23: "Environmental Sciences",
            31: "Chemistry",
            32: "Earth Sciences",
            33: "Physics & Astronomy",
            34: "Other Physical Sci",
            41: "Economics",
            42: "Political Sciences",
            43: "Psychology",
            44: "Sociology & Anthro",
            45: "Other Social Sci",
            51: "Aerospace Eng",
            52: "Chemical Eng",
            53: "Civil Eng",
            54: "Electrical & Comp Eng",
            55: "Industrial Eng",
            56: "Mechanical Eng",
            57: "Other Eng",
            61: "Health",
            62: "Sci & Math Education",
            63: "Technology & Tech",
            64: "Other S&E Related",
            71: "Management & Admin",
            72: "Education (non-S&E)",
            73: "Social Services",
            74: "Sales & Marketing",
            75: "Art & Humanities",
            76: "Other Non-S&E"
        }

        # Set display options
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)

        trans_matrix = pd.crosstab(
            df['NBAMENG'], 
            df['NDGMENG'],
            normalize='index'  # row percentages
        )

        # Rename both index and columns
        trans_matrix.index = trans_matrix.index.map(field_names)
        trans_matrix.columns = trans_matrix.columns.map(field_names)

        print(f"{title} (Row percentages)")
        print(trans_matrix.round(3))
        return trans_matrix
    
    # Field patterns
    def field_patterns(df, title):
        patterns = df.groupby('NBAMENG').agg({
            'field_switch': ['mean', 'count'],
            'NDGMENG': lambda x: x.mode()[0] if not x.empty else None
        }).round(3)
        
        print(f"\nField Patterns - {title}")
        print("Switch Rate = Proportion changing fields")
        print("Count = Number of observations")
        print("Most Common Destination = Most common master's field if switched")
        print(patterns)
        return patterns
    
    matrix_immediate = create_transition_matrix(immediate, "Immediate Master's")
    matrix_all = create_transition_matrix(all_masters, "All Master's")
    
    patterns_immediate = field_patterns(immediate, "Immediate Master's")
    patterns_all = field_patterns(all_masters, "All Master's")

    # Unemployment rate analysis (rest of the code remains the same)
    def ur_analysis(df, title):
        print(f"\nUnemployment Rate Analysis - {title}")
        ur_quintiles = pd.qcut(df['UR'], q=5)
        switch_by_ur = df.groupby(ur_quintiles)['field_switch'].mean()
        print("Switching rates by unemployment rate quintile:")
        print(switch_by_ur)

        # Statistical test
        from scipy import stats
        high_ur = df[df['UR'] > df['UR'].median()]['field_switch']
        low_ur = df[df['UR'] <= df['UR'].median()]['field_switch']
        t_stat, p_val = stats.ttest_ind(high_ur, low_ur)
        print(f"\nt-test p-value for high vs low unemployment: {p_val:.4f}")

    ur_analysis(immediate, "Immediate Master's")
    ur_analysis(all_masters, "All Master's")
    

    return {
        'matrix_immediate': matrix_immediate,
        'matrix_all': matrix_all,
        'patterns_immediate': patterns_immediate,
        'patterns_all': patterns_all
    }

In [ ]:
immediate, all_masters = prepare_data_minor(df_final)
results = descriptive_analysis_minor(immediate, all_masters)

In [ ]:
def analyze_ur_cohort_switch_minor(immediate, all_masters, title=None):
    """
    Analyze and visualize relationship between UR, field switching, and graduation cohort
    for either immediate master's or all master's recipients
    """
    # Calculate mean UR and switch rate by cohort
    stats_by_cohort = immediate.groupby('BAYR').agg({
        'UR': 'mean',
        'field_switch': 'mean'
    }).reset_index()
    
    # Create figure with two y-axes
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()
    
    # Plot unemployment rate on left y-axis
    line1 = ax1.plot(stats_by_cohort['BAYR'], stats_by_cohort['UR'], 
                     color='blue', marker='o', linestyle='-', linewidth=2,
                     label='Unemployment Rate')
    
    # Plot switch rate on right y-axis
    line2 = ax2.plot(stats_by_cohort['BAYR'], stats_by_cohort['field_switch'], 
                     color='red', marker='s', linestyle='-', linewidth=2,
                     label='Field Switch Rate')
    
    # Customize the plot
    ax1.set_xlabel('Bachelor Graduation Year', fontsize=10)
    ax1.set_ylabel('Unemployment Rate (%)', color='blue', fontsize=10)
    ax2.set_ylabel('Field Switch Rate', color='red', fontsize=10)
    
    title = title if title else 'Unemployment Rate and Field Switching Rate by Graduation Cohort'
    plt.title(title, fontsize=12, pad=15)
    
    # Add grid
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # Combine legends
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    
    # Print correlations
    ur_corr = immediate[['BAYR', 'UR']].corr().iloc[0,1]
    switch_corr = immediate[['BAYR', 'field_switch']].corr().iloc[0,1]
    ur_switch_corr = immediate[['UR', 'field_switch']].corr().iloc[0,1]
    
    print(f"\nCorrelations:")
    print(f"Graduation year - Unemployment rate: {ur_corr:.3f}")
    print(f"Graduation year - Switch rate: {switch_corr:.3f}")
    print(f"Unemployment rate - Switch rate: {ur_switch_corr:.3f}")
    
    plt.tight_layout()
    plt.show()
    
    return stats_by_cohort

# 分别对两组进行分析
# For immediate master's
trends_immediate_minor = analyze_ur_cohort_switch_minor(immediate, all_masters, 
                                          title="Immediate Master's: UR and Field Switching Rate")

# For all master's
trends_all_minor = analyze_ur_cohort_switch_minor(all_masters, all_masters, 
                                     title="All Master's: UR and Field Switching Rate")

In [ ]:
def analyze_field_flow_patterns(df, year):
    """
    Analyze field switching patterns for a specific graduation cohort using field descriptions
    """
    # Field name dictionary
    field_names = {
        11: "Computer & Info Sci",
        12: "Mathematics & Stats",
        21: "Agricultural Sciences",
        22: "Biological Sciences", 
        23: "Environmental Sciences",
        31: "Chemistry",
        32: "Earth Sciences",
        33: "Physics & Astronomy",
        34: "Other Physical Sci",
        41: "Economics",
        42: "Political Sciences",
        43: "Psychology",
        44: "Sociology & Anthro",
        45: "Other Social Sci",
        51: "Aerospace Eng",
        52: "Chemical Eng",
        53: "Civil Eng",
        54: "Electrical & Comp Eng",
        55: "Industrial Eng",
        56: "Mechanical Eng",
        57: "Other Eng",
        61: "Health",
        62: "Sci & Math Education",
        63: "Technology & Tech",
        64: "Other S&E Related",
        71: "Management & Admin",
        72: "Education (non-S&E)",
        73: "Social Services",
        74: "Sales & Marketing",
        75: "Art & Humanities",
        76: "Other Non-S&E"
    }
    
    # Filter for the specific cohort
    cohort_data = df[df['BAYR'] == year].copy()
    
    # Add field descriptions
    cohort_data['BA_field'] = cohort_data['NBAMENG'].map(field_names)
    cohort_data['MA_field'] = cohort_data['NDGMENG'].map(field_names)
    
    # Calculate total switches for this cohort
    n_switches = (cohort_data['field_switch'] == 1).sum()
    switch_rate = cohort_data['field_switch'].mean()
    
    # For those who switched (field_switch == 1)
    switchers = cohort_data[cohort_data['field_switch'] == 1]
    
    # Analyze outflows (source fields)
    outflows = switchers['BA_field'].value_counts()
    outflow_pcts = (outflows / n_switches * 100).round(1)
    
    # Analyze inflows (destination fields)
    inflows = switchers['MA_field'].value_counts()
    inflow_pcts = (inflows / n_switches * 100).round(1)
    
    # Get top 5 specific flow paths
    flow_paths = switchers.groupby(['BA_field', 'MA_field']).size()
    flow_paths_pcts = (flow_paths / n_switches * 100).round(1)
    top_paths = flow_paths_pcts.nlargest(5)
    
    return {
        'year': year,
        'n_switches': n_switches,
        'switch_rate': switch_rate,
        'top_outflows': outflow_pcts.head(),
        'top_inflows': inflow_pcts.head(),
        'top_paths': top_paths
    }

def analyze_switching_trends(df):
    """
    Analyze switching patterns across all cohorts
    """
    # Get all graduation years
    years = sorted(df['BAYR'].unique())
    
    # Analyze each cohort
    results = []
    for year in years:
        cohort_analysis = analyze_field_flow_patterns(df, year)
        results.append(cohort_analysis)
        
        print(f"\n{'='*80}")
        print(f"Graduation Year: {year}")
        print(f"Number of switches: {cohort_analysis['n_switches']}")
        print(f"Switch rate: {cohort_analysis['switch_rate']:.1%}")
        
        print("\nTop 5 Source Fields (% of switches):")
        print(cohort_analysis['top_outflows'])
        
        print("\nTop 5 Destination Fields (% of switches):")
        print(cohort_analysis['top_inflows'])
        
        print("\nTop 5 Specific Flow Paths (% of switches):")
        for (source, dest), pct in cohort_analysis['top_paths'].items():
            print(f"{source} → {dest}: {pct:.1f}%")
            
    return results

# # Run the analysis for both immediate and all masters
# print("\nIMMEDIATE MASTERS ANALYSIS")
# print("="*80)
# immediate_flow_results = analyze_switching_trends(immediate)

print("\nALL MASTERS ANALYSIS")
print("="*80)
all_masters_flow_results = analyze_switching_trends(all_masters)

In [ ]:
def analyze_field_flow_rates(df):
    """
    Analyze and visualize field-specific inflow and outflow rates across cohorts
    """
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    
    # Field name dictionary
    field_names = {
        11: "Computer & Info Sci",
        12: "Mathematics & Stats",
        21: "Agricultural Sciences",
        22: "Biological Sciences", 
        23: "Environmental Sciences",
        31: "Chemistry",
        32: "Earth Sciences",
        33: "Physics & Astronomy",
        34: "Other Physical Sci",
        41: "Economics",
        42: "Political Sciences",
        43: "Psychology",
        44: "Sociology & Anthro",
        45: "Other Social Sci",
        51: "Aerospace Eng",
        52: "Chemical Eng",
        53: "Civil Eng",
        54: "Electrical & Comp Eng",
        55: "Industrial Eng",
        56: "Mechanical Eng",
        57: "Other Eng",
        61: "Health",
        62: "Sci & Math Education",
        63: "Technology & Tech",
        64: "Other S&E Related",
        71: "Management & Admin",
        72: "Education (non-S&E)",
        73: "Social Services",
        74: "Sales & Marketing",
        75: "Art & Humanities",
        76: "Other Non-S&E"
    }
    
    # Add field descriptions if not already present
    if 'BA_field' not in df.columns:
        df['BA_field'] = df['NBAMENG'].map(field_names)
    if 'MA_field' not in df.columns:
        df['MA_field'] = df['NDGMENG'].map(field_names)
    
    # Get list of years
    years = sorted(df['BAYR'].unique())
    
    # Initialize storage for rates and field sizes
    inflow_rates = {}    # {field: {year: rate}}
    outflow_rates = {}   # {field: {year: rate}} 
    field_sizes = {}     # {field: {year: count}}
    all_fields = set(df['BA_field'].unique())
    
    # For each year, calculate field-specific rates
    for year in years:
        # Get cohort data
        cohort_data = df[df['BAYR'] == year].copy()
        
        # Count graduates by bachelor's field
        ba_counts = cohort_data['BA_field'].value_counts()
        
        # Store field sizes
        for field, count in ba_counts.items():
            if field not in field_sizes:
                field_sizes[field] = {}
            field_sizes[field][year] = count
        
        # Calculate outflow rates
        for field in all_fields:
            # Get graduates with this bachelor's field
            field_grads = cohort_data[cohort_data['BA_field'] == field]
            total_grads = len(field_grads)
            
            # Skip if no graduates in this field
            if total_grads == 0:
                continue
                
            # Count switchers (different master's field)
            switchers = field_grads[field_grads['field_switch'] == 1]
            
            # Calculate outflow rate: % of BA graduates who switch to different MA field
            outflow_rate = len(switchers) / total_grads * 100
            
            # Store rate
            if field not in outflow_rates:
                outflow_rates[field] = {}
            outflow_rates[field][year] = outflow_rate
        
        # Calculate inflow rates
        # Get all switchers
        switchers = cohort_data[cohort_data['field_switch'] == 1]
        
        # Count inflows by destination field
        inflows = switchers['MA_field'].value_counts()
        
        for field in all_fields:
            # Get total bachelor's graduates in this field
            ba_count = ba_counts.get(field, 0)
            
            # Skip if no graduates
            if ba_count == 0:
                continue
                
            # Count inflows (people switching INTO this field)
            inflow_count = inflows.get(field, 0)
            
            # Calculate inflow rate: inflows as % of field's bachelor's size
            inflow_rate = inflow_count / ba_count * 100
            
            # Store rate
            if field not in inflow_rates:
                inflow_rates[field] = {}
            inflow_rates[field][year] = inflow_rate
    
    # Calculate total field size to identify largest fields
    field_totals = {}
    for field, year_counts in field_sizes.items():
        field_totals[field] = sum(year_counts.values())
    
    # Select top fields for visualization (to avoid cluttering)
    top_fields = sorted(field_totals.items(), key=lambda x: x[1], reverse=True)[:8]
    top_field_names = [f[0] for f in top_fields]
    
    # Plot field-specific trends
    fig, axes = plt.subplots(len(top_field_names), 1, figsize=(15, 24), sharex=True)
    
    for i, field in enumerate(top_field_names):
        ax = axes[i]
        
        # Plot outflow rate
        outflow_data = [outflow_rates.get(field, {}).get(y, 0) for y in years]
        ax.plot(years, outflow_data, 'r-', marker='o', linewidth=2,
                label='Outflow Rate (% who switch out)')
        
        # Plot inflow rate 
        inflow_data = [inflow_rates.get(field, {}).get(y, 0) for y in years]
        ax.plot(years, inflow_data, 'g-', marker='s', linewidth=2,
                label='Inflow Rate (inflows as % of field size)')
        
        # Calculate and plot net flow
        net_flow = [i - o for i, o in zip(inflow_data, outflow_data)]
        ax.plot(years, net_flow, 'b--', marker='^', 
                label='Net Flow (inflow - outflow)')
        
        # Add zero reference line
        ax.axhline(y=0, color='k', linestyle=':', alpha=0.5)
        
        # Add field size on secondary axis for context
        ax2 = ax.twinx()
        size_data = [field_sizes.get(field, {}).get(y, 0) for y in years]
        ax2.bar(years, size_data, alpha=0.2, color='gray', 
                label='Number of BA graduates')
        ax2.set_ylabel('Number of graduates', color='gray')
        
        # Set up primary axis
        ax.set_title(f'{field}', fontsize=16)
        ax.set_ylabel('Rate (%)')
        ax.grid(True, alpha=0.3)
        
        # Combine legends
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    # Add common labels and title
    plt.xlabel('Bachelor Graduation Year', fontsize=14)
    plt.suptitle('Field-Specific Flow Rates Across Graduation Cohorts', 
                fontsize=20, y=0.95)
    
    # Add national unemployment rate for reference
    if 'UR' in df.columns:
        # Calculate average UR by year
        ur_by_year = df.groupby('BAYR')['UR'].mean()
        
        # Add small subplot for UR
        ur_ax = fig.add_axes([0.15, 0.92, 0.7, 0.05])
        ur_ax.plot(years, [ur_by_year.get(y, None) for y in years], 
                  'k-', marker='o', linewidth=2)
        ur_ax.set_title('National Unemployment Rate (%)', fontsize=14)
        ur_ax.grid(True, alpha=0.3)
        ur_ax.set_xticks([])
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.91)
    plt.show()
    
    return inflow_rates, outflow_rates, field_sizes

# Usage
inflow_rates, outflow_rates, field_sizes = analyze_field_flow_rates(all_masters)

In [ ]:
def analyze_field_flow_rates(df):
    """
    Analyze and visualize field-specific inflow and outflow rates across cohorts for all fields
    """
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    
    # Field name dictionary
    field_names = {
        11: "Computer & Info Sci",
        12: "Mathematics & Stats",
        21: "Agricultural Sciences",
        22: "Biological Sciences", 
        23: "Environmental Sciences",
        31: "Chemistry",
        32: "Earth Sciences",
        33: "Physics & Astronomy",
        34: "Other Physical Sci",
        41: "Economics",
        42: "Political Sciences",
        43: "Psychology",
        44: "Sociology & Anthro",
        45: "Other Social Sci",
        51: "Aerospace Eng",
        52: "Chemical Eng",
        53: "Civil Eng",
        54: "Electrical & Comp Eng",
        55: "Industrial Eng",
        56: "Mechanical Eng",
        57: "Other Eng",
        61: "Health",
        62: "Sci & Math Education",
        63: "Technology & Tech",
        64: "Other S&E Related",
        71: "Management & Admin",
        72: "Education (non-S&E)",
        73: "Social Services",
        74: "Sales & Marketing",
        75: "Art & Humanities",
        76: "Other Non-S&E"
    }
    
    # Add field descriptions if not already present
    if 'BA_field' not in df.columns:
        df['BA_field'] = df['NBAMENG'].map(field_names)
    if 'MA_field' not in df.columns:
        df['MA_field'] = df['NDGMENG'].map(field_names)
    
    # Get list of years
    years = sorted(df['BAYR'].unique())
    
    # Initialize storage for rates and field sizes
    inflow_rates = {}    # {field: {year: rate}}
    outflow_rates = {}   # {field: {year: rate}} 
    field_sizes = {}     # {field: {year: count}}
    all_fields = set(df['BA_field'].unique())
    
    # For each year, calculate field-specific rates
    for year in years:
        # Get cohort data
        cohort_data = df[df['BAYR'] == year].copy()
        
        # Count graduates by bachelor's field
        ba_counts = cohort_data['BA_field'].value_counts()
        
        # Store field sizes
        for field, count in ba_counts.items():
            if field not in field_sizes:
                field_sizes[field] = {}
            field_sizes[field][year] = count
        
        # Calculate outflow rates
        for field in all_fields:
            # Get graduates with this bachelor's field
            field_grads = cohort_data[cohort_data['BA_field'] == field]
            total_grads = len(field_grads)
            
            # Skip if no graduates in this field
            if total_grads == 0:
                continue
                
            # Count switchers (different master's field)
            switchers = field_grads[field_grads['field_switch'] == 1]
            
            # Calculate outflow rate: % of BA graduates who switch to different MA field
            outflow_rate = len(switchers) / total_grads * 100
            
            # Store rate
            if field not in outflow_rates:
                outflow_rates[field] = {}
            outflow_rates[field][year] = outflow_rate
        
        # Calculate inflow rates
        # Get all switchers
        switchers = cohort_data[cohort_data['field_switch'] == 1]
        
        # Count inflows by destination field
        inflows = switchers['MA_field'].value_counts()
        
        for field in all_fields:
            # Get total bachelor's graduates in this field
            ba_count = ba_counts.get(field, 0)
            
            # Skip if no graduates
            if ba_count == 0:
                continue
                
            # Count inflows (people switching INTO this field)
            inflow_count = inflows.get(field, 0)
            
            # Calculate inflow rate: inflows as % of field's bachelor's size
            inflow_rate = inflow_count / ba_count * 100
            
            # Store rate
            if field not in inflow_rates:
                inflow_rates[field] = {}
            inflow_rates[field][year] = inflow_rate
    
    # Calculate total field size
    field_totals = {}
    for field, year_counts in field_sizes.items():
        field_totals[field] = sum(year_counts.values())
    
    # Filter out fields with very small sample sizes for meaningful visualization
    valid_fields = [field for field, total in field_totals.items() 
                   if total >= 50 and field is not None]  # Minimum 50 graduates
    
    # Sort fields by total size for organized presentation
    sorted_fields = sorted([(field, field_totals.get(field, 0)) 
                           for field in valid_fields],
                          key=lambda x: x[1], reverse=True)
    sorted_field_names = [f[0] for f in sorted_fields]
    
    # Create multiple figures with 4 fields per figure for better readability
    fields_per_figure = 4
    num_figures = (len(sorted_field_names) + fields_per_figure - 1) // fields_per_figure
    
    # Add national unemployment rate as reference
    if 'UR' in df.columns:
        ur_by_year = df.groupby('BAYR')['UR'].mean()
    
    # Generate figures
    for fig_idx in range(num_figures):
        start_idx = fig_idx * fields_per_figure
        end_idx = min(start_idx + fields_per_figure, len(sorted_field_names))
        current_fields = sorted_field_names[start_idx:end_idx]
        
        fig, axes = plt.subplots(len(current_fields), 1, figsize=(15, 5*len(current_fields)), 
                                sharex=True)
        
        # Handle case where there's only one field in the figure
        if len(current_fields) == 1:
            axes = [axes]
        
        for i, field in enumerate(current_fields):
            ax = axes[i]
            
            # Plot outflow rate
            outflow_data = [outflow_rates.get(field, {}).get(y, 0) for y in years]
            ax.plot(years, outflow_data, 'r-', marker='o', linewidth=2,
                    label='Outflow Rate (% who switch out)')
            
            # Plot inflow rate 
            inflow_data = [inflow_rates.get(field, {}).get(y, 0) for y in years]
            ax.plot(years, inflow_data, 'g-', marker='s', linewidth=2,
                    label='Inflow Rate (inflows as % of field size)')
            
            # Calculate and plot net flow
            net_flow = [i - o for i, o in zip(inflow_data, outflow_data)]
            ax.plot(years, net_flow, 'b--', marker='^', 
                    label='Net Flow (inflow - outflow)')
            
            # Add zero reference line
            ax.axhline(y=0, color='k', linestyle=':', alpha=0.5)
            
            # Add field size on secondary axis for context
            ax2 = ax.twinx()
            size_data = [field_sizes.get(field, {}).get(y, 0) for y in years]
            ax2.bar(years, size_data, alpha=0.2, color='gray', 
                    label='Number of BA graduates')
            ax2.set_ylabel('Number of graduates', color='gray')
            
            # Set up primary axis
            ax.set_title(f'{field} (Total BA graduates: {field_totals[field]})', fontsize=14)
            ax.set_ylabel('Rate (%)')
            ax.grid(True, alpha=0.3)
            
            # Set y-axis limits to accommodate all data with some padding
            max_rate = max(max(outflow_data), max(inflow_data), max(net_flow)+10)
            min_rate = min(min(net_flow)-10, 0)
            ax.set_ylim(min_rate, max_rate)
            
            # Combine legends
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
        
        # Add common labels and title
        plt.xlabel('Bachelor Graduation Year', fontsize=14)
        title_range = f"Fields {start_idx+1}-{end_idx} of {len(sorted_field_names)}"
        plt.suptitle(f'Field-Specific Flow Rates Across Graduation Cohorts\n{title_range}', 
                    fontsize=16, y=0.95)
        
        # Add national unemployment rate for reference
        if 'UR' in df.columns:
            # Add small subplot for UR
            ur_ax = fig.add_axes([0.15, 0.95, 0.7, 0.03])
            ur_ax.plot(years, [ur_by_year.get(y, None) for y in years], 
                      'k-', marker='o', linewidth=2)
            ur_ax.set_title('National Unemployment Rate (%)', fontsize=12)
            ur_ax.grid(True, alpha=0.3)
            ur_ax.set_xticks([])
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.92)
        plt.show()
    
    return inflow_rates, outflow_rates, field_sizes

# Usage
inflow_rates, outflow_rates, field_sizes = analyze_field_flow_rates(all_masters)